# N100 · 网格DP与路径状态

**目标：** 根据允许方向建立无环转移并处理障碍。

**先修：** N018, N098, N031。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 坐标状态；边界；路径计数；最小代价；逆向需求状态。

## 从问题到算法

网格DP的重点是依赖方向。向右向下走使网格成为DAG，路径数相加、最小成本取min。地牢需要反向思考：dp[r][c]不是累计血量，而是进入该格前保证生存所需的最小血量；离开所需减去本格收益后，至少还要1。

## 最小实现

**本章接口：** `unique_paths(m, n)`、`unique_paths_with_obstacles(grid)`、`min_path_sum(grid)`、`calculate_minimum_hp(dungeon)`

In [1]:
def unique_paths(m,n):
    if m<=0 or n<=0: return 0
    dp=[1]*n
    for _ in range(1,m):
        for c in range(1,n): dp[c]+=dp[c-1]
    return dp[-1]

def unique_paths_with_obstacles(grid):
    if not grid or not grid[0]: return 0
    dp=[0]*len(grid[0]); dp[0]=1
    for row in grid:
        for c,x in enumerate(row):
            if x: dp[c]=0
            elif c: dp[c]+=dp[c-1]
    return dp[-1]

def min_path_sum(grid):
    if not grid or not grid[0]: return 0
    dp=[float('inf')]*len(grid[0]); dp[0]=0
    for row in grid:
        for c,x in enumerate(row): dp[c]=min(dp[c],dp[c-1] if c else float('inf'))+x
    return dp[-1]

def calculate_minimum_hp(dungeon):
    if not dungeon or not dungeon[0]: return 1
    m,n=len(dungeon),len(dungeon[0]); dp=[float('inf')]*(n+1); dp[n-1]=1
    for r in range(m-1,-1,-1):
        for c in range(n-1,-1,-1): dp[c]=max(1,min(dp[c],dp[c+1])-dungeon[r][c])
    return dp[0]

## 正确性、前提与复杂度

每条路径的最后或下一步只有两个互斥方向，DAG顺序确保依赖已计算。地牢从终点向前归纳：进入血量h满足h≥1且h+格值≥下一格所需，故h=max(1,next−格值)。

**代价：** O(mn)算术操作、O(n)辅助空间。矩形输入；教学扩展的空网格结果已显式定义。路径计数是任意精度整数，不隐式取模。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

grid=[[-2,-3,3],[-5,-10,1],[10,30,-5]]; dp=[[0]*3 for _ in range(3)]
for r in range(2,-1,-1):
    for c in range(2,-1,-1):
        after=1 if (r,c)==(2,2) else min([dp[nr][nc] for nr,nc in [(r+1,c),(r,c+1)] if nr<3 and nc<3])
        dp[r][c]=max(1,after-grid[r][c])
show_table(['行','进入各格最少血量'],list(enumerate(dp)))

行,进入各格最少血量
0,"[7, 5, 2]"
1,"[6, 11, 5]"
2,"[1, 1, 6]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert unique_paths(3,7)==28
assert unique_paths_with_obstacles([[1]])==0
assert unique_paths_with_obstacles([[0,0],[0,1]])==0
assert calculate_minimum_hp([[-5]])==6
assert calculate_minimum_hp([[-2,-3,3],[-5,-10,1],[10,30,-5]])==7
from itertools import product

def paths(g,r=0,c=0):
    if r==len(g)-1 and c==len(g[0])-1: return [[g[r][c]]]
    return [[g[r][c]]+tail for nr,nc in [(r+1,c),(r,c+1)] if nr<len(g) and nc<len(g[0]) for tail in paths(g,nr,nc)]
for a in product((-2,1,3),repeat=6):
    g=[a[:3],a[3:]]; p=paths(g)
    assert min_path_sum(g)==min(map(sum,p))
    need=min(max(1,1-min(sum(x[:k]) for k in range(1,len(x)+1))) for x in p)
    assert calculate_minimum_hp(g)==need
print('N100: 所有本章断言通过')

N100: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 对比最小路径和与最低初始生命值的状态。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 滚动到一行。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 62. Unique Paths（canonical）
- 63. Unique Paths II（canonical）
- 64. Minimum Path Sum（canonical）
- 174. Dungeon Game（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。